# Wood density one subsample for figure 1

In [1]:
import ee
import pandas as pd
import os
import numpy as np
import random
from random import sample
import itertools 
import geopandas as gpd
from sklearn.metrics import r2_score
from termcolor import colored # this is allocate colour and fonts type for the print title and text
from IPython.display import display, HTML

In [12]:
#set the working directory of local drive for Grid search result table loading
# os.getcwd()
os.chdir('D:/MaxTreeHeightProject/TreeHeightMap/TreeStructureProject')

In [13]:
# initialize the earth engine API
ee.Initialize(project='ee-sun520ying')

## STEP 1 Data preperation and objects definition

In [14]:
# Load the composite for root shoot ratio analysis
compositeRaw = ee.Image("users/leonidmoore/ForestBiomass/20200915_Forest_Biomass_Predictors_Image")
# get the projection
stdProj = compositeRaw.projection()
forestAgeData = ee.Image("users/leonidmoore/ForestBiomass/20251218_ForestAge_SoilMoisture_Image")
print(colored('Band in Forest age data:\n', 'blue', attrs=['bold']),forestAgeData.bandNames().getInfo())
# we choose the band "forest_age_TC030" for following modeling
# forestAge = forestAgeData.select(['ForestAge']).reproject(crs=stdProj).rename('ForestAge')
# soilmoisture = forestAgeData.select(['SoilMoisture']).reproject(crs=stdProj).rename('SoilMoisture')
# load the additional layers and uniform the projection
# soilmoisture = ee.Image('users/haozhima95/wld_soil_moisture').reproject(crs=stdProj).rename('SoilMoisture')
waterAvail = compositeRaw.select("CHELSA_Annual_Precipitation").subtract(compositeRaw.select("PET")).rename("WaterAvailability")
ET_Ver3 = ee.Image('users/leonidmoore/Evapotranspiration_0_Ver_3_2023').rename('ET0').reproject(stdProj)
CHELSA_vpd = ee.Image("projects/ee-sun520ying/assets/CHELSA_vpd_mean_1981-2010_V_2_1").rename('CHELSA_vpd')
compositeImg = compositeRaw.addBands(forestAgeData).addBands(ET_Ver3).addBands(waterAvail).addBands(CHELSA_vpd)
# check the band names in the composite
compositeBandNames = compositeImg.bandNames()
print(colored('Band in composite:\n', 'blue', attrs=['bold']),compositeBandNames.getInfo())

Band in Forest age data:
 ['ForestAge', 'SoilMoisture']
Band in composite:
 ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvCloudCover_MODCF_interannualSD', 'EarthEnvCloudCover_MODCF_intraannualSD', 'EarthEnvCloudCover_MODCF_meanannual', 'EarthEnvTopoMe

In [15]:
# define the list of predictors
predictorsSelected = ["Aridity_Index",
                      "CHELSA_Annual_Mean_Temperature",
                      "CHELSA_Annual_Precipitation",
                      "CHELSA_Isothermality",
                      "CHELSA_Max_Temperature_of_Warmest_Month",
                      "CHELSA_Mean_Diurnal_Range",
                      "CHELSA_Mean_Temperature_of_Coldest_Quarter",
                      "CHELSA_Mean_Temperature_of_Driest_Quarter",
                      "CHELSA_Mean_Temperature_of_Warmest_Quarter",
                      "CHELSA_Mean_Temperature_of_Wettest_Quarter",
                      "CHELSA_Min_Temperature_of_Coldest_Month",
                      "CHELSA_Precipitation_Seasonality",
                      "CHELSA_Precipitation_of_Coldest_Quarter",
                      "CHELSA_Precipitation_of_Driest_Month",
                      "CHELSA_Precipitation_of_Driest_Quarter",
                      "CHELSA_Precipitation_of_Warmest_Quarter",
                      "CHELSA_Precipitation_of_Wettest_Month",
                      "CHELSA_Precipitation_of_Wettest_Quarter",
                      "CHELSA_Temperature_Annual_Range",
                      "CHELSA_Temperature_Seasonality",
                      "Depth_to_Water_Table",
                      "EarthEnvCloudCover_MODCF_interannualSD",
                      "EarthEnvCloudCover_MODCF_intraannualSD",
                      "EarthEnvCloudCover_MODCF_meanannual",
                      "EarthEnvTopoMed_Eastness",
                      "EarthEnvTopoMed_Elevation",
                      "EarthEnvTopoMed_Northness",
                      "EarthEnvTopoMed_ProfileCurvature",
                      "EarthEnvTopoMed_Roughness",
                      "EarthEnvTopoMed_Slope",
                      "EarthEnvTopoMed_TopoPositionIndex",
                      "SG_Absolute_depth_to_bedrock",
                      "WorldClim2_SolarRadiation_AnnualMean",
                      "WorldClim2_WindSpeed_AnnualMean",
                      "WorldClim2_H2OVaporPressure_AnnualMean",
                      "NDVI",
                      "EVI",
                      "Lai",
                      "Fpar",
                      "Npp",
                      "Tree_Density",
                      "PET",
                      "SG_Clay_Content_0_100cm",
                      "SG_Coarse_fragments_0_100cm",
                      "SG_Sand_Content_0_100cm",
                      "SG_Silt_Content_0_100cm",
                      "SG_Soil_pH_H2O_0_100cm",
                      "LandCoverClass_Cultivated_and_Managed_Vegetation",
                      "LandCoverClass_Urban_Builtup",
                      "Human_Disturbance",
                      "PresentTreeCover",
                      "Nitrogen",
                      "cropland",
                      "grazing",
                      "pasture",
                      "rangeland",
                      "cnRatio",
                      "Cation",
                      "SoilMoisture",
                      "ForestAge",
                      "Organic_Carbon",
                      "GlobBiomass_GrowingStockVolume",
                      "Human_Development_Percentage",
                      "HumanFootprint",
                      "Rainfall_Erosivity",
                      "WDPA",
                      "Fire_Frequency",
                      "WaterAvailability",
                      'CHELSA_vpd']
# show the predictors
print(colored('The predictors are:', 'blue', attrs=['bold']),predictorsSelected)

The predictors are: ['Aridity_Index', 'CHELSA_Annual_Mean_Temperature', 'CHELSA_Annual_Precipitation', 'CHELSA_Isothermality', 'CHELSA_Max_Temperature_of_Warmest_Month', 'CHELSA_Mean_Diurnal_Range', 'CHELSA_Mean_Temperature_of_Coldest_Quarter', 'CHELSA_Mean_Temperature_of_Driest_Quarter', 'CHELSA_Mean_Temperature_of_Warmest_Quarter', 'CHELSA_Mean_Temperature_of_Wettest_Quarter', 'CHELSA_Min_Temperature_of_Coldest_Month', 'CHELSA_Precipitation_Seasonality', 'CHELSA_Precipitation_of_Coldest_Quarter', 'CHELSA_Precipitation_of_Driest_Month', 'CHELSA_Precipitation_of_Driest_Quarter', 'CHELSA_Precipitation_of_Warmest_Quarter', 'CHELSA_Precipitation_of_Wettest_Month', 'CHELSA_Precipitation_of_Wettest_Quarter', 'CHELSA_Temperature_Annual_Range', 'CHELSA_Temperature_Seasonality', 'Depth_to_Water_Table', 'EarthEnvCloudCover_MODCF_interannualSD', 'EarthEnvCloudCover_MODCF_intraannualSD', 'EarthEnvCloudCover_MODCF_meanannual', 'EarthEnvTopoMed_Eastness', 'EarthEnvTopoMed_Elevation', 'EarthEnvTopoM

In [16]:
# Define a function to take a feature with a classifier of interest
def computeCVAccuracy(featureWithClassifier,
                      propertyOfInterest,
                      modelType,
                      kFoldAssignmentFC,
                      cvFoldString,
                      classProperty,
                      extractedVariableTable):
    # Pull the classifier from the feature
    cOI = ee.Classifier(featureWithClassifier)
    # Create a function to map through the fold assignments and compute the overall accuracy
    # for all validation folds
    def computeAccuracyForFold(foldFeature):
        # Organize the training and validation data
        foldNumber = ee.Number(ee.Feature(foldFeature).get('Fold'))
        # print(foldNumber.getInfo())
        trainingData = extractedVariableTable.filterMetadata(cvFoldString,'not_equals',foldNumber)
        # print(trainingData.first().getInfo())
        validationData = extractedVariableTable.filterMetadata(cvFoldString,'equals',foldNumber)
        # Train the classifier and classify the validation dataset
        trainedClassifier = cOI.train(trainingData,classProperty,propertyOfInterest)
        outputtedPropName = 'Predicted'
        classifiedValidationData = validationData.classify(trainedClassifier,outputtedPropName)
        # Create a central if/then statement that determines the type of accuracy values that are returned
        copyProps = ['x','y','MaxHeight','CV_Fold']
        return classifiedValidationData.copyProperties(source = trainingData,properties = copyProps) #.set('Fold',ee.Number(foldNumber))

    # Compute the accuracy values of the classifier across all folds
    accuracyFC = kFoldAssignmentFC.map(computeAccuracyForFold).flatten()
    
    return accuracyFC

In [18]:
cvFoldString = 'CV_Fold'
classProperty = 'MaxHeight'
# define a loop through the seed list
seedList = np.arange(0, 50, 1).tolist()
print(colored('The models are:', 'blue', attrs=['bold']),seedList)
print(colored('Model is running:\nWith paramter sets:', 'blue', attrs=['bold']))
# for seed in seedList: range(0,len(seedList))
for seed in seedList:
    # load the points data in shape file format
    woodDensityPoints = ee.FeatureCollection('users/sun520ying/TreeStructureProject/TrainTables/TreeHeight_filtered_BufferZone_20260407_Subsampled_Train_table_Seed_'+str(seed))
    # check the information of the FeatureCollection
    # print(woodDensityTable.first().getInfo())
    # extract the environment covariates
    extractedVariableTable = compositeImg.select(predictorsSelected).reduceRegions(collection = woodDensityPoints,
                                                                                     reducer = ee.Reducer.first())
    # show the str of the extracted data
    # print(extractedVariableTable.first().getInfo())
    # exclude the rows with null valus
    trainTable = extractedVariableTable.filter(ee.Filter.notNull(predictorsSelected))
    # print(trainTable.size().getInfo())
    parameterTable = pd.read_csv('D:/MaxTreeHeightProject/TreeHeightMap/TreeStructureProject/ModelOptimization/GridSearchResultsGEE/Tree_Height_filtered_Buffer_Based_20260407_Subsample_Grid_Search_Seed_'+str(seed)+'.csv', float_precision='round_trip')
    # not recomend to run the code below
    # print(parameterTable.head())
    # extract the paramters
    variablesPerSplitVal = int(parameterTable['variablesPerSplit'].iat[0]) # mtry
    minLeafPopulationVal = int(parameterTable['minLeafPopulation'].iat[0]) # minrow
    maxNodesVal = int(parameterTable['maxNodes'].iat[0]) # mac depth
    print('seed',seed,variablesPerSplitVal,minLeafPopulationVal,maxNodesVal)
    # define the random forest classifier
    rfClassifier = ee.Classifier.smileRandomForest(numberOfTrees = 500,
                                                   variablesPerSplit = variablesPerSplitVal, # mtry
                                                   minLeafPopulation = minLeafPopulationVal, # minrow
                                                   maxNodes = maxNodesVal, # max depth
                                                   bagFraction = 0.632,
                                                   seed = seed).setOutputMode('REGRESSION')
    kList = list(range(0,10))
    kFoldAssignmentFC = ee.FeatureCollection(ee.List(kList).map(lambda n: ee.Feature(ee.Geometry.Point([0,0])).set('Fold',n)))
    
    predictedAndObs = computeCVAccuracy(rfClassifier,predictorsSelected,modelType='REGRESSION',kFoldAssignmentFC= kFoldAssignmentFC,cvFoldString = cvFoldString,classProperty=classProperty,extractedVariableTable = trainTable)
    # select the target property names
    predObs = predictedAndObs.select(['x','y','MaxHeight','Predicted','CV_Fold'])
    # transfer the feature collection into list for further export
    to_export = predObs.toList(5000).getInfo()
    result = []
    for item in to_export:
        values = item['properties']
        row = [str(values[key]) for key in ['x','y','MaxHeight','Predicted','CV_Fold']]
        row = ",".join(row)
        result.append(row)
    # for loop to write each feature as a row into local folder
    df = pd.DataFrame([item.split(",") for item in result], columns = ['x','y','MaxHeight','Predicted','CV_Fold'])
    with open('D:/MaxTreeHeightProject/TreeHeightMap/TreeStructureProject/PredictAndObs/Exported_Table_of_Prediction_and_Observation_with_10_fold_for_'+str(seed)+'.csv', 'a') as f:
        df.to_csv(f, mode='a', header=f.tell()==0)
    # show the progress
    print(colored('Prediction of model'+str(seed)+' finished!', 'blue', attrs=['bold']))

The models are: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
Model is running:
With paramter sets:
seed 0 15 2 90
Prediction of model0 finished!
seed 1 18 2 80
Prediction of model1 finished!
seed 2 9 4 80
Prediction of model2 finished!
seed 3 21 2 100
Prediction of model3 finished!
seed 4 12 4 100
Prediction of model4 finished!
seed 5 9 2 100
Prediction of model5 finished!
seed 6 15 4 100
Prediction of model6 finished!
seed 7 18 4 100
Prediction of model7 finished!
seed 8 18 2 100
Prediction of model8 finished!
seed 9 9 2 100
Prediction of model9 finished!
seed 10 21 4 80
Prediction of model10 finished!
seed 11 21 2 100
Prediction of model11 finished!
seed 12 21 4 60
Prediction of model12 finished!
seed 13 21 2 90
Prediction of model13 finished!
seed 14 21 2 100
Prediction of model14 finished!
seed 15 6 4 100
Prediction of model15 finished!
s